# 01 — Semantic Prime Hashing (C1: `token → γ`)

**Fourth Age Paper companion notebook.** `ScalarContextPropagation` — component **C1**.

C1 is the address step: a word's spelling, deterministically, becomes an
index into the non-trivial Riemann zeros, which selects **which pencil**
(which of the 7 box-kite charts) the word's box kite belongs to. Nothing
here is a claim under test — it is the live pipeline, run and checked.

    H(w)  = Σ_k ord(w_k) · 95^(|w|−1−k)     base-95 Horner, offset 32
    p     = next_prime(H(w) mod 2^16)       prime in [2, 65537]
    idx   = π(p)                            ∈ [1, 6543]  (see the anomaly chase below —
                                             the paper currently states 6542)
    γ     = the idx-th non-trivial Riemann zero (Im), Z(t) Newton

Provenance: **OURS** — `VAPMIP/monad.py` (`_horner_hash`, `_next_prime`,
`_word_zero_idx`, `_gamma_at`), `VAPMIP/monad_bin/SPEC.md §3`.

This notebook is the scaffold for the rigorous testing pass on this
algorithm — collision rate, determinism, bucket distribution, and the one
known live bug (a tier-0/tier-1 category error) are all measured directly
against the live 347,119-word store, not asserted.

In [1]:
import sys, os, time
sys.path.insert(0, os.path.expanduser("~/Projects/ThePlace/VAPMIP"))
import monad

print("PRIME_CAP:", monad._PRIME_CAP, " max idx (pi(PRIME_CAP)):", monad._prime_pi_table[monad._PRIME_CAP])
assert monad._prime_pi_table[monad._PRIME_CAP] == 6542, "6542 pencils claim depends on this"


PRIME_CAP: 65536  max idx (pi(PRIME_CAP)): 6542


## The pass, step by step, on one real word

In [2]:
def trace(w: str):
    h = monad._horner_hash(w)
    p = monad._next_prime(h)
    idx = monad._word_zero_idx(w)
    g = monad._gamma_at(idx)
    print(f"{w!r:>14}  H={h:>22}  next_prime={p:>6}  idx={idx:>5}  gamma={g:.6f}")
    return dict(word=w, horner=h, prime=p, idx=idx, gamma=g)

for w in ["accretion", "pile", "toroidal", "condensate", "narrative", "dissertation"]:
    trace(w)


   'accretion'  H=    435952031571538408  next_prime=  7933  idx= 1002  gamma=1439.015472


        'pile'  H=              69256114  next_prime= 49043  idx= 5042  gamma=5548.516421
    'toroidal'  H=      5924746544723326  next_prime= 13513  idx= 1601  gamma=2117.754191
  'condensate'  H=  42756310149759033674  next_prime=  3761  idx=  523  gamma=853.143566
   'narrative'  H=    522068046232455314  next_prime=  1847  idx=  283  gamma=522.366924
'dissertation'  H=391207708623858272995283  next_prime= 46919  idx= 4847  gamma=5365.310518


## Determinism

The address must be a pure function of spelling — same input, same output,
every call, no hidden state.

In [3]:
import random
words_sample = ["accretion", "pile", "toroidal", "condensate", "narrative",
                "dissertation", "kite", "octahedron", "wind", "tension"]
ok = True
for w in words_sample:
    a = monad._word_zero_idx(w)
    b = monad._word_zero_idx(w)
    c = monad._word_zero_idx(w)
    if not (a == b == c):
        ok = False
        print("NON-DETERMINISTIC:", w, a, b, c)
print("all repeated calls identical:", ok)


all repeated calls identical: True


## Collision rate and bucket distribution over the live vocabulary

`monad3_c.bin` holds 347,119 words. This is a 16-bit hash into 6542
buckets (`idx`) — **not** claimed injective by the paper (see gate **G2**
in the README): many words legitimately share a pencil, and resolution
within a pencil is the wind speed `w`'s job, not this address's. What
*is* being measured here is the actual distribution shape and whether the
known live bug reproduces.

In [4]:
import struct, mmap

STORE = os.path.expanduser("~/Projects/ThePlace/VAPMIP/PtolC/monad3_c.bin")

def iter_words(path, limit=None):
    f = open(path, "rb")
    mm = mmap.mmap(f.fileno(), 0, access=mmap.ACCESS_READ)
    hdr = struct.Struct("<8s6I16d13Q")
    vals = hdr.unpack_from(mm, 0)
    magic, ver, n_words, n_eng, n_wn, n_phon, nnz = vals[:7]
    assert magic.rstrip(b"\x00") == b"MONAD3C", magic
    off = vals[23:]
    o_blob, o_rec = off[0], off[1]
    rec = struct.Struct("<iiii")
    def name(o):
        e = mm.find(b"\x00", o_blob + o)
        return mm[o_blob + o:e].decode("utf-8", "replace")
    n = n_words if limit is None else min(limit, n_words)
    for i in range(n):
        noff, ei, wi, pi = rec.unpack_from(mm, o_rec + i * 16)
        yield name(noff)
    return

t0 = time.time()
words = list(iter_words(STORE))
print(f"loaded {len(words):,} words in {time.time()-t0:.2f}s")


loaded 347,119 words in 0.34s


In [5]:
from collections import Counter

t0 = time.time()
buckets = Counter()
idx_of = {}
for w in words:
    idx = monad._word_zero_idx(w)
    buckets[idx] += 1
    idx_of[w] = idx
dt = time.time() - t0
print(f"hashed {len(words):,} words in {dt:.2f}s  ({len(words)/dt:,.0f} words/s)")
print(f"idx range: [{min(buckets)}, {max(buckets)}]  distinct buckets used: {len(buckets)}/6542")
print(f"words per bucket: min {min(buckets.values())}  max {max(buckets.values())}  "
      f"mean {len(words)/len(buckets):.1f}")

top = buckets.most_common(5)
print("most-populated buckets:", top)


hashed 347,119 words in 3.17s  (109,403 words/s)
idx range: [1, 6543]  distinct buckets used: 6543/6542
words per bucket: min 1  max 395  mean 53.1
most-populated buckets: [(3386, 395), (3645, 338), (4523, 310), (5009, 306), (5950, 302)]


## The known live bug — tier-0 category error

`_horner_hash` clamps digits from below only (`max(0, ord(ch) - 32)`), so
an invisible Unicode character (e.g. U+200B, zero-width space) that maps
to the same clamped digit as a printable tier-1 sequence collides with
it. This is flagged in `VAPMIP/prime_hash.py`'s design notes as a live bug,
not fixed here — reproduced directly below so the notebook carries the
measurement, not just the claim.

In [6]:
zwsp = "\u200b"
h1 = monad._horner_hash(zwsp)
h2 = monad._horner_hash("v!")
print(f"horner(U+200B) = {h1}   horner('v!') = {h2}   equal: {h1 == h2}")
print(f"idx(U+200B) = {monad._word_zero_idx(zwsp)}   idx('v!') = {monad._word_zero_idx('v!')}")
print()
print("STATUS: reproduced. This is a tier-0/tier-1 boundary error (whitespace/control")
print("characters are APERTURE, not LETTERS — see prime_hash.py) and remains open.")


horner(U+200B) = 8171   horner('v!') = 8171   equal: True
idx(U+200B) = 1026   idx('v!') = 1026

STATUS: reproduced. This is a tier-0/tier-1 boundary error (whitespace/control
characters are APERTURE, not LETTERS — see prime_hash.py) and remains open.


## Anomaly, chased — the range is `[1, 6543]`, not `[1, 6542]`

The bucket scan above found `idx = 6543` — one past the paper's own stated
range. Chased to a verdict rather than rounded away:

- `next_prime` is documented (and coded) to reach **65537** inclusive.
- 65537 is itself prime, so `π(65537) = π(65536) + 1 = 6543`.
- The paper states two things that cannot both be exactly true: `p ∈ [2,
  65537]` **and** `idx ∈ [1, 6542]`. If 65537 is reachable, 6543 is too.

**Verdict: MATHS/METHOD, not CODE.** The code is internally consistent (its
`π` table is sized for exactly this case) — it is the paper's two stated
ranges that disagree with each other. This is not a corner case nobody
hits: 91 real words in the live vocabulary land in bucket 6543, including
ordinary English words (`apocope`, `contemplating`, `discipline`).

In [7]:
hit_6543 = [w for w in words if monad._word_zero_idx(w) == 6543]
print(f"{len(hit_6543)} words hit idx 6543, e.g.:", sorted(hit_6543)[:12])
print()
print("pi(65536) =", monad._prime_pi_table[65536], " pi(65537) =", monad._prime_pi_table[65537])
print("=> there are 6543 distinct pencils reachable, not 6542.")


91 words hit idx 6543, e.g.: ["'kay", '*face', '1387–1422', '1}{x}}\\right', '2024-07-09', '_{0}+|\\sigma', 'a)\\subsetneq', 'acetaldol', 'allograft', 'apocope', 'ben-david', 'bʰarant-']

pi(65536) = 6542  pi(65537) = 6543
=> there are 6543 distinct pencils reachable, not 6542.


## Summary

| check | result |
|---|---|
| determinism (same word → same idx, repeated calls) | confirmed |
| collision behaviour | non-injective by design (16-bit hash into buckets); rate and shape measured above, not asserted |
| U+200B / `'v!'` collision (documented live bug) | reproduced |
| **6542 vs. 6543 pencils** | **anomaly found**: `idx` reaches 6543 because `next_prime` can return 65537 itself, which is prime. The paper's `[2, 65537]` for `p` and `[1, 6542]` for `idx` are mutually inconsistent; the correct range is `idx ∈ [1, 6543]`. Flagged for a README fix, not silently patched here. |

This notebook is the harness for further rigorous testing of C1 — extend
the cells above with whatever specific property is under test next.